# RAG Implementation

To run this notebook successfully, a Groq API key and Ollama running locally is required. 

Groq:
- Put the Groq API key as `GROQ_API_KEY=...` in the `.env` file

Ollama:
- Install Ollama and run `ollama serve`
- Install the embedding model with `ollama pull nomic-embed-text`

In [1]:
# Imports
import pandas as pd
import numpy as np
import requests
import csv
from groq import Groq
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

In [2]:
# Load environment variables from .env file
load_dotenv()

True

In [3]:
# Ollama endpoint
OLLAMA = "http://localhost:11434"

# Models
EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "openai/gpt-oss-20b"

## Embedding and LLM Setup

First we need to embed the documents we want to use for RAG. Then, we can send the embeddings along with our query to the LLM.

In [4]:
# Function to send requests to embed model
def embed(texts):
    r = requests.post(f"{OLLAMA}/api/embed", json={"model": EMBED_MODEL, "input": texts})
    return np.array(r.json()["embeddings"])

In [5]:
# Function to send requests to LLM model
def ask_llm(prompt):
    client = Groq()

    try:
        r = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return r.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")

In [6]:
ask_llm("What is the capital of France?")

'The capital of France is **Paris**.'

## Process and Load in Data

Now we'll want to load in the recall data and perform some data cleaning. Load the data into a DataFrame first, then convert the DataFrame rows into a formatted text document.

In [7]:
# Load data in a DataFrame
DATA_PATH = Path("../data/raw/FLAT_RCL_POST_2010.txt") # notebook lives in notebooks/

COLUMNS = [
    "RECORD_ID", "CAMPNO", "MAKETXT", "MODELTXT", "YEARTXT", "MFGCAMPNO",
    "COMPNAME", "MFGNAME", "BGMAN", "ENDMAN", "RCLTYPECD", "POTAFF", "ODATE",
    "INFLUENCED_BY", "MFGTXT", "RCDATE", "DATEA", "RPNO", "FMVSS",
    "DESC_DEFECT", "CONEQUENCE_DEFECT", "CORRECTIVE_ACTION", "NOTES",
    "RCL_CMPT_ID", "MFR_COMP_NAME", "MFR_COMP_DESC", "MFR_COMP_PTNO",
    "DO_NOT_DRIVE", "PARK_OUTSIDE"
]

df = pd.read_csv(
    DATA_PATH,
    sep="\t", # Data is stored as .txt but is separated by tabs
    header=None,           
    names=COLUMNS,
    dtype=str,             
    encoding="latin-1",
    quoting=csv.QUOTE_NONE,
    keep_default_na=False,
)

In [8]:
df.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,DO_NOT_DRIVE,PARK_OUTSIDE
0,81716,10V484000,CARRIAGE,CAMEO,2005,,EQUIPMENT,"CARRIAGE, INC",,,...,CARRIAGE IS RECALLING CERTAIN RECREATIONAL VEH...,AN OVERHEATED RECEIVER COULD CAUSE A FIRE.,CARRIAGE IS WORKING WITH DIMPLEX AND DIMPLEX W...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000037779000206087000000329,,,,No,No
1,81717,10V484000,CARRIAGE,CAMEO,2006,,EQUIPMENT,"CARRIAGE, INC",,,...,CARRIAGE IS RECALLING CERTAIN RECREATIONAL VEH...,AN OVERHEATED RECEIVER COULD CAUSE A FIRE.,CARRIAGE IS WORKING WITH DIMPLEX AND DIMPLEX W...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000037779000768768000000329,,,,No,No
2,81718,10V484000,CARRIAGE,CAMEO,2007,,EQUIPMENT,"CARRIAGE, INC",,,...,CARRIAGE IS RECALLING CERTAIN RECREATIONAL VEH...,AN OVERHEATED RECEIVER COULD CAUSE A FIRE.,CARRIAGE IS WORKING WITH DIMPLEX AND DIMPLEX W...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000037779000768769000000329,,,,No,No
3,81719,10V484000,CARRIAGE,CAMEO,2003,,EQUIPMENT,"CARRIAGE, INC",,,...,CARRIAGE IS RECALLING CERTAIN RECREATIONAL VEH...,AN OVERHEATED RECEIVER COULD CAUSE A FIRE.,CARRIAGE IS WORKING WITH DIMPLEX AND DIMPLEX W...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000037779000201426000000329,,,,No,No
4,81720,10V144000,GLAVAL,APOLLO,2007,,EQUIPMENT ADAPTIVE/MOBILITY,GLAVAL BUS DIVISION,20071001,20091231,...,GLAVAL BUS IS RECALLING CERTAIN MODEL YEAR 200...,WHEN THE PLATFORM OVERRUNS THE VEHICLE FLOOR L...,GLAVAL BUS WILL NOTIFY OWNERS AND THE REPAIRS ...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000035679000338665000001004,,,,No,No


In [9]:
# Build chunks from rows of the DataFrame
def build_doc(g):
    first = g.iloc[0]
    models = sorted(set(g["MAKETXT"] + " " + g["MODELTXT"] + " " + g["YEARTXT"]))
    text = (
        f"Recall campaign {first['CAMPNO']}.\n"
        f"Manufacturer: {first['MFGNAME']}.\n"
        f"Component: {first['COMPNAME']}.\n"
        f"Affected vehicles: {'; '.join(models)}.\n"
        f"Defect: {first['DESC_DEFECT']}\n"
        f"Consequence: {first['CONEQUENCE_DEFECT']}\n"
        f"Remedy: {first['CORRECTIVE_ACTION']}"
    )
    return {
        "id": first["CAMPNO"],
        "text": text,
        "makes": sorted(set(g["MAKETXT"])),
        "years": sorted(set(g["YEARTXT"])),
        "component": first["COMPNAME"],
    }

docs = [build_doc(g) for _, g in df.groupby("CAMPNO")]

In [10]:
# Embed each chunk's `text` (turn chunks into vectors)
embeddings_path = Path("../data/processed/embeddings.npy")
texts = ["search_document: " + doc["text"] for doc in docs]

# Skip redoing embeddings if embeddings data already exists
if embeddings_path.is_file():
    chunk_vectors = np.load(embeddings_path)
else:
    chunk_vectors = np.vstack([embed(texts[i:i+64]) for i in range(0, len(texts), 64)])
    np.save("../data/processed/embeddings.npy", chunk_vectors) # Avoid re-embedding on every restart

    print(f"Indexed {len(docs)} chunks")

## Searching / Retrieval
With the chunk vectors, we now want to search through them based on similarity to a query. Take the user's question and turn it into a vector as well, then compare it to each chunk with cosine similarity. Return the top *k* most similar chunks to the user query.

In [11]:
# Function to retrieve the top k most similar chunks to a query
def retrieve(query, k=5):
    q = embed([query])[0]
    scores = chunk_vectors @ q / (np.linalg.norm(chunk_vectors, axis=1) * np.linalg.norm(q))
    return [texts[i] for i in np.argsort(scores)[::-1][:k]]

## Calling the LLM with RAG

Attach the retrieved chunks as context to the user's query and send it to the LLM.

In [12]:
def ask_llm_rag(question):
    context = "\n\n".join(retrieve(question))
    prompt = f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know."

Context:
{context}

Question: {question}"""

    return ask_llm(prompt)

In [13]:
question = "What is the recall for the 2015 Ford F-150?"

response = ask_llm_rag(question)
display(Markdown(response))

Ford is recalling certain 2015 F‑150 trucks for three separate problems:

| Recall | Component | What can go wrong | Risk | What Ford will do |
|--------|-----------|------------------|------|-------------------|
| **15V279000** | Steering – upper intermediate shaft | The shaft may be riveted improperly, letting it separate from the flex‑coupling | Loss of steering control → higher crash risk | Dealers will inspect the shaft and replace it if needed, free of charge |
| **15V251000** | Structure – underbody heat shields | Heat shields may be loose or missing | Fuel/vapor lines can degrade or melt → fire risk | Dealers will check for missing shields/fasteners and install them at no cost |
| **15C17** (15V867000) | Seat belts – front anchorage | Pretensioner cable may not be crimped correctly | Seat belts may fail to restrain occupants in a crash → higher injury risk | Dealers will replace the affected retractor(s) and pretensioner(s) for free |

Ford will notify owners of the affected vehicles (manufactured in the specified date ranges in 2015). Owners can contact Ford customer service at **1‑866‑436‑7332** for details.

In [14]:
# Chat loop (q to quit)
while True:
    question = input("Ask a question: ")

    if question.lower() == "q":
        break
    
    display(Markdown(f"**Question:** {question}"))

    response = ask_llm_rag(question)
    display(Markdown(response))

**Question:** What are the Tesla Model Y recalls?

**Tesla Model Y recalls (as listed in the provided context)**  

| Recall campaign (number) | Component/issue | Affected Model Y years | Key defect | Consequence | Remedy |
|--------------------------|-----------------|------------------------|------------|-------------|--------|
| SB‑26‑19‑002 (26V315000) | Certification label | 2025, 2026 | Label may not have been installed, missing weight information | Unintentionally overloading the vehicle, higher crash risk | Inspect and install label free of charge |
| SB‑21‑31‑004 (21V912000) | Suspension (front & rear knuckles) | 2020, 2021, 2022 | Knuckles may fracture, suspension links separate | Reduced stability/control, higher crash risk | Inspect and replace knuckles free of charge |
| SB‑20‑31‑012 (20V709000) | Suspension – front upper control arm & steering knuckle | 2020 | Bolts may not be tightened, arm may detach | Wheels may lean, reduced steering, higher crash risk | Inspect and tighten bolts free of charge |
| SB‑22‑18‑002 (22V050000) | Electrical system – software (heat‑pump valve) | 2020, 2021, 2022 | Software error may open valve, trapping refrigerant | Decreased defrosting → reduced visibility, higher crash risk | OTA software update free of charge |
| SB‑23‑32‑001 (23V385000) | Steering – critical fasteners | 2022, 2023 | Steering wheel fastener may be loose | Steering wheel may disconnect, loss of steering control, higher crash risk | Inspect and replace fastener free of charge |

These are all the recall campaigns in the context that involve Tesla Model Y vehicles.

**Question:** Has the Toyota Camry ever been recalled?

Yes. The context lists several recall campaigns that affected Toyota Camry vehicles, such as the 2018 Camry with engine and fuel‑system issues, the 2011 Camry with suspension concerns, and Camry models from 2023–2026 with seat and instrument‑cluster problems.

**Question:** What are the Honda Civic recalls?

**Honda Civic recall campaigns (based on the provided documents)**  

| Recall campaign | Vehicle(s) involved | Defect / issue | Remedy |
|-----------------|---------------------|----------------|--------|
| **18V421000** – “Certification label” issue | 2018 Civic (any model) | Random characters on the manufacturing‑date area of certification labels can prevent owners from determining if a safety recall applies to their vehicle | Honda will notify owners and dealers will replace the certification label free of charge |
| **18V817000** – “Owner’s guide” issue | 2017–2018 Civic Hatchback, 2018 Civic Type R | Owner’s guide may be missing or not contain required information, so the vehicle fails to meet FMVSS 225 (Child Restraint Anchorage Systems) | Honda will notify owners and dealers will replace the owner’s information kit free of charge |
| **17V706000** – “Half‑shaft” issue | 2017 Civic Sedan, 2017 Civic Coupe | Right half‑shaft may have been improperly heat‑treated, reducing its strength | Honda will notify owners and dealers will inspect the half‑shaft serial number and replace it if necessary, then align the front wheels free of charge |
| **12V548000** – “Steering column” issue | 2012 Civic (manufactured Oct 26‑Oct 30, 2012) | Incorrect steering‑column assembly may lack proper energy‑absorbing characteristics | Honda will notify owners and dealers will inspect and replace the steering column free of charge |
| **11V106000** – “Voltage‑converter” issue | 2006–2007 Civic Hybrid | Voltage converter that relays power from the IMA system may fail, causing headlights to turn off, engine stall, and inability to restart | Dealers will replace the voltage converter free of charge |

These are all the recall campaigns listed in the context for Honda Civic vehicles.

**Question:** How many times has the Toyota Prius been recalled?

I don't know.

**Question:** What is the capital of France?

I don't know.